# Actividad 1: Arquitectura de Redes Neuronales

## Dataset Sonar

El objetivo es clasificar una señal de sonar como **piedra (R)** o **mina (M)** usando una red neuronal.

La solución sigue los puntos solicitados y mantiene una estructura simple, similar a los ejemplos de la carpeta RA1.

## 1. Análisis exploratorio de datos

El archivo no tiene nombres de columnas. Las primeras 60 columnas son señales de entrada y la última columna es la clase:

- `R`: piedra.
- `M`: mina.

In [ ]:
# Importar módulos
import time
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías para el preprocesamiento y la evaluación
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

: 

In [ ]:
# Cargar el dataset
# El notebook debe ejecutarse desde la carpeta Actividad-1
sonar = pd.read_csv('sonar data.csv', header=None)

# Crear nombres simples para las 60 señales y la variable de salida
columnas = [f'frecuencia_{i}' for i in range(1, 61)] + ['clase']
sonar.columns = columnas

print(f'Tamaño del dataset: {sonar.shape}')
sonar.head()

### Revisión general

In [ ]:
# Estadísticas básicas de las variables numéricas
sonar.describe()

In [ ]:
# Verificar espacios en blanco, valores NaN y filas duplicadas
print('Valores NaN por columna:')
print(sonar.isnull().sum().sum())
print('Filas duplicadas:', sonar.duplicated().sum())
print('Clases encontradas:', sonar['clase'].unique())
print('Tipos de datos:')
print(sonar.dtypes.value_counts())

In [ ]:
# Revisar el rango de las señales
entradas = sonar.drop('clase', axis=1)
print('Valor mínimo:', entradas.min().min())
print('Valor máximo:', entradas.max().max())
print('Valores no numéricos:', entradas.apply(pd.to_numeric, errors='coerce').isnull().sum().sum())

In [ ]:
# Revisar si el dataset está balanceado
conteo_clases = sonar['clase'].value_counts()
porcentaje_clases = sonar['clase'].value_counts(normalize=True).mul(100).round(2)

print(conteo_clases)
print('Porcentajes:')
print(porcentaje_clases)

conteo_clases.plot(kind='bar', color=['steelblue', 'darkorange'])
plt.title('Cantidad de muestras por clase')
plt.xlabel('Clase')
plt.ylabel('Cantidad')
plt.xticks(rotation=0)
plt.show()

**Resultado del balance:** hay 111 muestras de minas (M) y 97 de piedras (R). La distribución es aproximadamente 53.37% y 46.63%, por lo que el dataset está suficientemente equilibrado, aunque no es perfectamente 50/50.

In [ ]:
# Detectar posibles outliers usando el rango intercuartílico (IQR)
Q1 = entradas.quantile(0.25)
Q3 = entradas.quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR
mascara_outliers = (entradas.lt(limite_inferior, axis=1) | entradas.gt(limite_superior, axis=1))

print('Posibles valores outlier:', mascara_outliers.sum().sum())
print('Filas con al menos un posible outlier:', mascara_outliers.any(axis=1).sum())
print('Variables con al menos un posible outlier:', mascara_outliers.any(axis=0).sum())

# Boxplot simple para observar la dispersión de las señales
plt.figure(figsize=(15, 4))
sns.boxplot(data=entradas, color='lightblue')
plt.title('Distribución de las 60 señales')
plt.xlabel('Variables de entrada')
plt.ylabel('Valor de señal')
plt.xticks([])
plt.show()

El criterio IQR puede marcar valores alejados del centro, pero no significa que sean errores. En este dataset las señales están dentro del rango válido 0 a 1, no hay NaN, no hay duplicados y no se observan valores imposibles. Por eso no se eliminan automáticamente los posibles outliers.

## 2. Preprocesamiento

Se separan las entradas `X` y la salida `y`. La clase se transforma a números:

- `R` se transforma en 0.
- `M` se transforma en 1.

Después se separan los datos en entrenamiento y testeo usando una división 75/25 y manteniendo la proporción de clases.

In [ ]:
# Separar X e y
X = sonar.drop('clase', axis=1).values.astype('float32')
y = sonar['clase'].map({'R': 0, 'M': 1}).values.astype('float32')

print(f'Conjunto X: {X.shape}')
print(f'Conjunto y: {y.shape}')

# Separar entrenamiento y testeo
X_e, X_t, y_e, y_t = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=40
)

print(f'X_e: {X_e.shape} - y_e: {y_e.shape}')
print(f'X_t: {X_t.shape} - y_t: {y_t.shape}')

In [ ]:
# Estandarizar usando solamente el conjunto de entrenamiento
escalador = StandardScaler()
X_e = escalador.fit_transform(X_e)
X_t = escalador.transform(X_t)

print('Promedio aproximado de X_e:', X_e.mean().round(4))
print('Desviación estándar aproximada de X_e:', X_e.std().round(4))

## 3. Diseño de la arquitectura

Se propone una red pequeña, suficiente para un dataset de solo 208 observaciones:

- Entrada: 60 neuronas, una por cada señal.
- Primera capa oculta: 32 neuronas con activación ReLU.
- Segunda capa oculta: 16 neuronas con activación ReLU.
- Dropout de 20% para ayudar a reducir el sobreajuste.
- Salida: 1 neurona con activación sigmoide, porque se trata de una clasificación binaria.
- Función de pérdida: `binary_crossentropy`.
- Solver/optimizador: Adam con ratio de aprendizaje 0.001.
- Métrica principal: accuracy.

## 4. Implementación con Keras/TensorFlow

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Dejar resultados reproducibles
np.random.seed(40)
tf.random.set_seed(40)

modelo = keras.Sequential([
    layers.Input(shape=(60,)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.20),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

modelo.summary()

## 5. Entrenamiento y tiempo de ejecución

Se entrenará durante 200 épocas, con lotes de 16 observaciones. El conjunto de test se usa como conjunto de validación para visualizar su evolución, tal como se muestra en los ejemplos de RA1.

In [ ]:
# Información de la máquina de entrenamiento
print('Sistema operativo:', platform.platform())
print('Procesador:', platform.processor())
print('Versión de Python:', platform.python_version())
print('Versión de TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

# Entrenamiento y medición del tiempo
inicio = time.perf_counter()

historial = modelo.fit(
    X_e,
    y_e,
    epochs=200,
    batch_size=16,
    validation_data=(X_t, y_t),
    verbose=0
)

fin = time.perf_counter()
print(f'Tiempo de entrenamiento: {fin - inicio:.2f} segundos')

## 6. Evaluación del modelo

Se visualiza la evolución de la pérdida y de la precisión para entrenamiento y testeo. Luego se calcula la matriz de confusión y el reporte de clasificación.

In [ ]:
# Evolución de la pérdida
plt.figure(figsize=(10, 4))
plt.plot(historial.history['loss'], 'r', label='Loss entrenamiento')
plt.plot(historial.history['val_loss'], 'b', label='Loss testeo')
plt.title('Evolución de la pérdida')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Evolución de la precisión
plt.figure(figsize=(10, 4))
plt.plot(historial.history['accuracy'], 'r', label='Accuracy entrenamiento')
plt.plot(historial.history['val_accuracy'], 'b', label='Accuracy testeo')
plt.title('Evolución de la precisión')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# Evaluación final en entrenamiento y testeo
perdida_e, precision_e = modelo.evaluate(X_e, y_e, verbose=0)
perdida_t, precision_t = modelo.evaluate(X_t, y_t, verbose=0)

print(f'Precisión entrenamiento: {precision_e * 100:.2f}%')
print(f'Precisión testeo: {precision_t * 100:.2f}%')
print(f'Pérdida entrenamiento: {perdida_e:.4f}')
print(f'Pérdida testeo: {perdida_t:.4f}')

In [ ]:
# Predicciones y matriz de confusión
probabilidades = modelo.predict(X_t, verbose=0)
predicciones = (probabilidades >= 0.5).astype(int).ravel()

print('Accuracy calculada:', accuracy_score(y_t, predicciones))
print('Matriz de confusión:')
print(confusion_matrix(y_t, predicciones))
print('Reporte de clasificación:')
print(classification_report(y_t, predicciones, target_names=['R - Piedra', 'M - Mina']))

## 7. Análisis de resultados

El valor final de accuracy de testeo se obtiene en la celda anterior. Como referencia, una precisión cercana o superior al 80% representa una mejora importante frente al clasificador básico de aproximadamente 53%, y se acerca al rango esperado para este dataset.

La arquitectura se considera generalizable si la precisión de testeo es alta y no existe una diferencia excesiva entre entrenamiento y testeo. La siguiente celda entrega una conclusión automática usando una diferencia máxima de 10 puntos porcentuales.

In [ ]:
diferencia = abs(precision_e - precision_t)
print(f'Diferencia entre entrenamiento y testeo: {diferencia * 100:.2f} puntos porcentuales')

if precision_t >= 0.80 and diferencia <= 0.10:
    print('Conclusión: el modelo logra una precisión esperada y muestra una generalización razonable.')
elif precision_t >= 0.80:
    print('Conclusión: el modelo logra una precisión esperada, pero presenta señales de sobreajuste.')
else:
    print('Conclusión: el modelo todavía no alcanza la precisión esperada y se podría ajustar la arquitectura o los hiperparámetros.')

### Conclusión general

El dataset está prácticamente balanceado y no contiene datos faltantes, duplicados ni valores fuera del rango válido de las señales. Se utilizaron 60 entradas, dos capas ocultas y una salida binaria. La red se implementó con Keras/TensorFlow y se midieron el tiempo de entrenamiento, la evolución de la pérdida, la precisión, la matriz de confusión y el reporte de clasificación.